# EOT 3D Adversarial Attack
**Athalye et al. 2017 — Synthesizing Robust Adversarial Examples**

Przed uruchomieniem: `Runtime → Change runtime type → T4 GPU`

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU: ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'brak')
print('PyTorch:', torch.__version__)

## 1. Instalacja PyTorch3D
Koła dobrane do wersji PyTorcha na Colabie.

In [ ]:
import sys, torch

# Dobierz koło pytorch3d do wersji torch+CUDA na tym Colabie
version_str = ''.join([
    f"py3{sys.version_info.minor}_cu",
    torch.version.cuda.replace('.', ''),
    "_pyt",
    torch.__version__.replace('.', '').split('+')[0]
])
print('Szukam koła dla:', version_str)

!pip install --quiet fvcore iopath
!pip install --quiet --no-index --no-deps \
    --find-links https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/{version_str} \
    pytorch3d 2>/dev/null || \
pip install --quiet "git+https://github.com/facebookresearch/pytorch3d.git@stable"

import pytorch3d
print('PyTorch3D:', pytorch3d.__version__)

## 2. Wgraj pliki projektu
Wgraj folder z GitHuba lub ręcznie.

In [ ]:
# Opcja A: sklonuj repo (jeśli jest na GitHubie)
# !git clone https://github.com/TWOJE_REPO/AI-Sec-Team6.git
# %cd AI-Sec-Team6

# Opcja B: wgraj ręcznie przez panel Files po lewej stronie Colaba
# Potrzebujesz:
#   pipeline_3d/   (renderer.py, attack.py, transforms.py, __init__.py)
#   pipeline_2d/   (attack.py, transforms.py, __init__.py)
#   models/        (classifier.py, __init__.py)
#   turtle/        (20446_Sea_Turtle_v1 Textured.obj + .mtl + .jpg)

import os
print('Pliki w /content:', os.listdir('/content'))

In [ ]:
# Tworzy puste __init__.py — Colab nie wgrywa pustych plików
from pathlib import Path
for pkg in ['pipeline_3d', 'pipeline_2d', 'models']:
    Path(f'{pkg}/__init__.py').touch()
print('__init__.py gotowe')

## 3. Konfiguracja ataku

In [ ]:
# ── Ścieżka do mesha ──────────────────────────────────────────────────────────
OBJ_PATH = "turtle/20446_Sea_Turtle_v1 Textured.obj"

# ── Klasa docelowa ────────────────────────────────────────────────────────────
# 764 = rifle | 429 = baseball | 35 = turtle | 404 = airliner
TARGET_CLASS = 764

# ── Hiperparametry ────────────────────────────────────────────────────────────
NUM_STEPS    = 500   # kroków PGD
EOT_SAMPLES  = 40    # losowych widoków na krok
STEP_SIZE    = 0.01  # długość kroku
TEXTURE_SIZE = 256   # rozdzielczość mapy tekstury
IMAGE_SIZE   = 224   # rozmiar renderowanego obrazu

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 4. Ładowanie modelu i renderera

In [ ]:
from models.classifier import InceptionV3Classifier
from pipeline_3d.renderer import TexturedMeshRenderer
from pipeline_3d.transforms import TransformConfig, sample_transforms
from pipeline_3d.attack import EOTAttack3D, Attack3DConfig

classifier = InceptionV3Classifier(device=DEVICE)
renderer   = TexturedMeshRenderer(
    OBJ_PATH,
    texture_size=TEXTURE_SIZE,
    image_size=IMAGE_SIZE,
    device=DEVICE,
)
print('Texture shape:', renderer.texture_map.shape)
print('Backend: PyTorch3D' if 'Pytorch3D' in type(renderer).__name__ else 'Backend: Software renderer')

## 5. Klasyfikacja przed atakiem

In [ ]:
import matplotlib.pyplot as plt

transform_cfg = TransformConfig()

with torch.no_grad():
    R, T, bg = sample_transforms(6, transform_cfg, DEVICE)
    images, _ = renderer.render_with_background(R, T, bg)
    logits = classifier(images)
    preds  = logits.argmax(dim=1).tolist()
    probs  = torch.softmax(logits, dim=1)

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
fig.suptitle('Przed atakiem', fontsize=13)
for i, ax in enumerate(axes):
    ax.imshow(images[i].cpu().numpy().clip(0,1))
    ax.axis('off')
    ax.set_title(f'cls {preds[i]}\n{probs[i,preds[i]]:.1%}', fontsize=9)
plt.tight_layout()
plt.show()

## 6. Atak EOT 3D

In [ ]:
attack_cfg = Attack3DConfig(
    target_class = TARGET_CLASS,
    step_size    = STEP_SIZE,
    num_steps    = NUM_STEPS,
    eot_samples  = EOT_SAMPLES,
    log_every    = 50,
)

attack = EOTAttack3D(
    classifier    = classifier,
    renderer      = renderer,
    transform_cfg = transform_cfg,
    attack_cfg    = attack_cfg,
    device        = DEVICE,
)

best_texture, history = attack.attack()

## 7. Wyniki

In [ ]:
with torch.no_grad():
    R, T, bg = sample_transforms(6, transform_cfg, DEVICE)
    images_adv, _ = renderer.render_with_background(R, T, bg)
    logits_adv = classifier(images_adv)
    preds_adv  = logits_adv.argmax(dim=1).tolist()
    probs_adv  = torch.softmax(logits_adv, dim=1)

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
fig.suptitle(f'Po ataku (target={TARGET_CLASS})', fontsize=13)
for i, ax in enumerate(axes):
    ax.imshow(images_adv[i].cpu().numpy().clip(0,1))
    ax.axis('off')
    ax.set_title(f'cls {preds_adv[i]}\n{probs_adv[i,preds_adv[i]]:.1%}', fontsize=9)
plt.tight_layout()
plt.show()

success = sum(p == TARGET_CLASS for p in preds_adv)
print(f'Atak trafil: {success}/6 widokow')

In [ ]:
# Loss curve
plt.figure(figsize=(7, 3))
plt.plot(history['loss'])
plt.xlabel('Log step')
plt.ylabel('Loss')
plt.title('EOT 3D loss')
plt.tight_layout()
plt.show()

In [ ]:
# Adversarial texture
tex = best_texture.squeeze(0).cpu().numpy().clip(0, 1)
plt.figure(figsize=(4, 4))
plt.imshow(tex)
plt.axis('off')
plt.title('Adversarial texture')
plt.show()

# Zapis
import matplotlib
matplotlib.image.imsave('adv_texture.png', tex)
print('Zapisano: adv_texture.png')